# Hypothetical Path Creator


## Imports

In [12]:
import pandas as pd
from scipy.integrate import quad
import re
import numpy as np

## Reading Data (Table B1 and Table B2)

### Table B1

In [13]:
Table_B1 = pd.read_excel('IPT_Tabel_B1.xlsx').fillna(0)
Table_B1 = Table_B1.apply(lambda col: col.str.strip() if col.dtype == "object" else col) #remove spaces

# make numerical columns float
num_cols_B1 = ["Molaire massa [g/mol]", "SG (20°/4°)", "Tm [°C]", "Hm(Tm) [kJ/mol]", "Tb [°C]", "Hv(Tb) [kJ/mol]", "Tc [K]", "Pc [atm]"]
Table_B1[num_cols_B1] = Table_B1[num_cols_B1].apply(pd.to_numeric, errors="coerce")

# make Hc string
Table_B1["Hc° [kJ/mol]"] = Table_B1["Hc° [kJ/mol]"].replace("-", np.nan)


In [14]:
# Extra Pre-Processing for Hf and Hc

# Split Heat of Formation Values for Different Phases
for row in range(len(Table_B1)):
    string = Table_B1["Hf° [kJ/mol]"].iloc[row]
    string = string.replace('`', '')
    splits = string.split()
    for i in range(len(splits)):
        s = splits[i]
        s = re.match(r"(-?\d+\.?\d*)\((\w+)\)", s)
        if s:
            value = float(s.group(1))
            phase = str(s.group(2))
            Table_B1.loc[row, f"Hf° ({phase}) [kJ/mol]"] = value


# Split Heat of Combustion Values for Different Phases
for row in range(len(Table_B1)):
    string = Table_B1["Hc° [kJ/mol]"].iloc[row]

    if pd.isna(string):
        continue

    string = string.replace('`', '')
    splits = string.split()
    for i in range(len(splits)):
        s = splits[i]
        s = re.match(r"(-?\d+\.?\d*)\((\w+)\)", s)
        if s:
            value = float(s.group(1))
            phase = str(s.group(2))
            Table_B1.loc[row, f"Hc° ({phase}) [kJ/mol]"] = value

# Drop original Hc and Hf columns
Table_B1 = Table_B1.drop(['Hf° [kJ/mol]', 'Hc° [kJ/mol]'], axis=1)


Table_B1

,Stofnaam (NL),Stofnaam (EN),Formule,Molaire massa [g/mol],SG (20°/4°),Tm [°C],Hm(Tm) [kJ/mol],Tb [°C],Hv(Tb) [kJ/mol],Tc [K],Pc [atm],Hf° (l) [kJ/mol],Hf° (g) [kJ/mol],Hf° (aq) [kJ/mol],Hf° (c) [kJ/mol],Hc° (l) [kJ/mol],Hc° (g) [kJ/mol],Hc° (s) [kJ/mol],Hc° (c) [kJ/mol]
0,Aceton,Acetone,C3H6O,58.08,0.791,-95.00,5.690,56.00,30.200,508.0,47.0,-248.20,-216.70,NaN,NaN,-1785.7,-1821.40,NaN,NaN
1,Acetyleen / ethyn,Acetylene,C2H2,26.04,NaN,NaN,NaN,-81.50,17.600,309.5,61.6,NaN,226.75,NaN,NaN,NaN,-1299.60,NaN,NaN
2,Ammoniak,Ammonia,NH3,17.03,NaN,-77.80,5.653,-33.43,23.351,405.5,111.3,-67.20,-46.19,NaN,NaN,NaN,-382.58,NaN,NaN
3,Ammoniumhydroxide,Ammonium hydroxide,NH4OH,35.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-366.48,NaN,NaN,NaN,NaN,NaN
4,Ammoniumnitraat,Ammonium nitrate,NH4NO3,80.05,1.725,169.60,5.400,NaN,NaN,NaN,NaN,NaN,NaN,-399.36,365.14,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Zuurstof,Oxygen,O2,32.00,NaN,-218.75,0.444,-182.97,6.820,154.4,49.7,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN
128,Zwavel,Sulfur,S8,256.53,2.070,113.00,10.040,444.60,83.700,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN
129,Zwaveldioxide,Sulfur dioxide,SO2,64.07,NaN,-75.48,7.402,-10.02,24.910,430.7,77.8,NaN,-296.90,NaN,NaN,NaN,NaN,NaN,NaN
130,Zwaveltrioxide,Sulfur trioxide,SO3,80.07,NaN,16.84,25.480,43.30,41.800,491.4,83.8,NaN,-395.18,NaN,NaN,NaN,NaN,NaN,NaN


### Table B2

In [24]:
Table_B2 = pd.read_excel('IPT_Tabel_B2.xlsx')
Table_B2 = Table_B2.apply(lambda col: col.str.strip()) #remove spaces
Table_B2 = Table_B2.fillna(0) # fill NaN w 0

num_cols_B2 = ['Molaire massa', 'Vorm', 'a (•10^3)', 'b (•10^5)', 'c (•10^8)', 'd (•10^12)']

Table_B2[num_cols_B2] = Table_B2[num_cols_B2].apply(pd.to_numeric, errors="coerce")

Table_B2.head(10)

,Stofnaam (NL),Stofnaam (EN),Formule,Molaire massa,Staat,Vorm,Eenheid van temperatuur,a (•10^3),b (•10^5),c (•10^8),d (•10^12)
0,Aceton,Acetone,CH3COCH3,58.08,l,1,°C,123.00,18.600,0.0000,0.000
1,Aceton,Acetone,CH3COCH3,58.08,g,1,°C,71.96,20.100,-12.7800,34.760
2,Ethyne,Acetylene,C2H2,26.04,g,1,ºC,42.43,6.053,-5.0330,18.200
3,Ammoniak,Ammonia,NH3,17.03,g,1,°C,35.15,2.954,0.4421,-6.686
4,Ammoniumsulfaat,Ammonium sulfate,(NH4)2SO4,132.15,c,1,K,215.90,0.000,0.0000,0.000
5,Benzeen,Benzene,C6H6,78.11,l,1,ºC,126.50,23.400,0.0000,0.000
6,Benzeen,Benzene,C6H6,78.11,g,1,ºC,74.06,32.950,-25.2000,77.570
7,Butaan,n-Butane,C4H10,58.12,g,1,ºC,92.30,27.880,-15.4700,34.980
8,Calciumcarbide,Calcium carbide,CaC2,64.10,c,2,K,68.62,1.190,NaN,0.000
9,Calciumcarbonaat,Calcium carbonate,CaCO3,100.09,c,2,K,82.34,4.975,NaN,0.000


## Defining Current Molecule State and Properties

In [ ]:
class Molecule:
    def __init__(self, name, T, P, F, Table_B1, Table_B2):
        self.name = name
        
        self.T = T
        self.P = P
        self.F = F

        self.Table_B1 = Table_B1[(Table_B1["Stofnaam (NL)"] == self.name) | (Table_B1["Stofnaam (EN)"] == self.name) | (Table_B1["Formule"] == self.name)].iloc[0]       # Selects row of Table B1 for the correct compound
        self.Table_B2 = Table_B2[((Table_B2["Stofnaam (NL)"] == self.name) | (Table_B2["Stofnaam (EN)"] == self.name) | (Table_B2["Formule"] == self.name)) & (Table_B2["Staat"] == self.F)].iloc[0]         # Selects row of Table B2 for the correct compound and phase
    

    ### Temperature Change

    # Get Cp (not integrated yet!)
    def Cp(self, T = None):
        if T is None:
            T = self.T

        coeff = self.Table_B2
        a = coeff['a (•10^3)']*1e-3
        b = coeff['b (•10^5)']*1e-5
        c = coeff['c (•10^8)']*1e-8
        d = coeff['d (•10^12)']*1e-12
        if coeff['Vorm'] == 1:
            return  a + b*T + c*T**2 + d*T**3
        elif coeff['Vorm'] == 2:
            return a + b*T + c*T**(-2)
        else:
            raise ValueError("Cp function is not of Form 1 or Form 2")

    # Integrate CpdT between T boundaries
    def temp_change(self, new_T):
        dH = quad(lambda T: self.Cp(T), self.T, new_T)[0] #kJ/mol

        # Adjust to New Temperature
        self.T = new_T
        return dH
    

    ### Phase Change

    def phase_change(self, new_F):
        solid = 's' or 'c'      # idk why but solid is given as 'c'

        # Melting and Freezing
        if self.F == solid and new_F == 'l':
            dH = self.Table_B1['Hm(Tm) [kJ/mol]']
        elif self.F == 'l' and new_F == solid:
            dH = -self.Table_B1['Hm(Tm) [kJ/mol]']
        # Vaporization and Condensation
        elif self.F == 'l' and new_F == 'g':
            dH = self.Table_B1['Hv(Tb) [kJ/mol]']
        elif self.F == 'g' and new_F == 'l':
            dH = -self.Table_B1['Hv(Tb) [kJ/mol]']
        # No Phase Change
        elif self.F == new_F:
            dH = 0
        # Error for Other Changes
        else:
            raise ValueError('There is no enthalpy provided for this phase transition.')

        # Adjust to new phase
        self.F = new_F
        return dH

## Making Steps

In [17]:
class Step:
    def __init__(self, molecule, new_T = None, new_F = None, new_P = None):
        self.molecule = molecule

        self.current_T = molecule.T
        self.current_P = molecule.P
        self.current_F = molecule.F

        self.new_T = new_T or molecule.T
        self.new_P = new_P or molecule.P
        self.new_F = new_F or molecule.F

        self.dH = 0

        if self.current_T != self.new_T:
            self.dH += molecule.temp_change(new_T)
        
        if self.current_F != self.new_F:
            self.dH += molecule.phase_change(new_F)
        
    



## Putting Steps into Path

In [18]:
class Path:
    def __init__(self, molecule, end_T, end_P, end_F):
        self.steps = []

        self.molecule = molecule

        self.end_T = end_T
        self.end_P = end_P
        self.end_F = end_F

    def add_step(self, step):
        self.steps.append(step)

    def total_enthalpy(self):
        enthalpy = 0
        for step in self.steps:
            enthalpy += step.dH
        return enthalpy

    def build_path(self):
        # Go to Tmelt and Phase Change to Liquid
        if self.molecule.F in ['c', 's'] and self.end_F in ['l', 'g']:
            self.add_step(Step(self.molecule, new_T = self.molecule.Table_B1['Tm [°C]']))
            self.add_step(Step(self.molecule, new_F = 'l'))

        # Go to Tmelt and Phase Change to Solid
        if self.molecule.F == 'l' and self.end_F in ['c', 's']:
            self.add_step(Step(self.molecule, new_T = self.molecule.Table_B1['Tm [°C]']))
            self.add_step(Step(self.molecule, new_F = 'c'))

        # Go to Tvap and Phase Change to Liquid if Ending in Solid
        if self.molecule.F == 'g' and self.end_F in ['c', 's']:
            self.add_step(Step(self.molecule, new_T = self.molecule.Table_B1['Tb [°C]']))
            self.add_step(Step(self.molecule, new_F = 'l'))

        # Go to Tvap and Change Phase
        if {self.molecule.F, self.end_F} == {'l', 'g'}:
            self.add_step(Step(self.molecule, new_T = self.molecule.Table_B1['Tb [°C]']))
            if self.molecule.F == 'l' and self.end_F == 'g':
                self.add_step(Step(self.molecule, new_F='g'))
            elif self.molecule.F == 'g' and self.end_F == 'l':
                self.add_step(Step(self.molecule, new_F='l'))
    
        # Final Temp Change
        if self.molecule.F == self.end_F and self.molecule.T != self.end_T:
            self.add_step(Step(self.molecule, new_T = self.end_T))

        

## Checking


In [19]:
water = Molecule(
    name="Water",
    T=300,
    P=101325,
    F="g",
    Table_B1=Table_B1,
    Table_B2=Table_B2
)

cp_now = water.Cp()        # uses current temperature (300 K)
print(cp_now)

0.036111349


In [29]:
# Try it Out
name = 'CaC2'
F_start = 'c'
F_end = 'c'
T_start = 400 # C
T_end = 500 # C
P_start = 1  # atm
P_end = 1    # atm

# Create molecule
water = Molecule(name, T_start, P_start, F_start, Table_B1, Table_B2)
path = Path(water, T_end, P_end, F_end)
path.build_path()
print(path.total_enthalpy())

nan


C:\Users\elise\AppData\Local\Temp\ipykernel_59044\3905735773.py:34: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  dH = quad(lambda T: self.Cp(T), self.T, new_T)[0] #kJ/mol
